In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\01_Foundation\\structured_output')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\01_Foundation\structured_output


In [2]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [3]:
from openai import OpenAI
openai_client  = OpenAI()

Testing the OPENAI Pydantic function example

In [4]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [5]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [11]:
#The OpenAI SDK has built-in support for structured output. Let's use the example from the documentation: 
# Alice and Bob are going to a science fair on Friday.

response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

In [7]:
response.output[0].content[0].parsed

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [8]:
response.output[0].content[0].text

'{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

In [12]:
event = response.output_parsed

In [13]:
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [ ]:
# the reason we use response.parse instead of response.create is that the former will automatically parse the output into the specified format, 
# while the latter will return the raw text output from the model.The format is inconsistent

#For example the below code might give different output each time you run it


response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


RECAP of RAG function we implemented in the previous lesson

In [14]:
##Going back to the document RAG
##RAG Recap

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")


Indexed 385 chunks from 95 documents


In [15]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results


In [17]:
#Define the build_prompt function:

import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )




In [20]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [18]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [21]:
answer = rag('how do I implement LLM as a judge?')

In [22]:
answer

'To implement an LLM as a judge, follow these steps based on the tutorial:\n\n1. **Prerequisites**:\n   - Ensure you have basic Python knowledge.\n   - Obtain an OpenAI API key for the LLM evaluator.\n   - It\'s recommended to run the tutorial in Jupyter Notebook or Google Colab for better visualization.\n\n2. **Installation and Setup**:\n   - Install the Evidently library:\n     ```bash\n     pip install evidently\n     ```\n   - Import the necessary modules:\n     ```python\n     import pandas as pd\n     import numpy as np\n     from evidently import Dataset, DataDefinition, Report, BinaryClassification\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     ```\n   - Set your OpenAI key as an environment variable:\n     ```python\n     import os\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n     ```\n\n3. **Create the Dataset**:\n   - Generate a toy Q&A dataset that includes questions, target responses, new responses, and manual labels indicating if th

Incorporate the Structured Output in the LLM function

In [ ]:
## Now let's try to use structured output to get a more consistent output format from the model.
#  We will define a Pydantic model for the expected output and use the parse method to get the structured output.
## we will modify the llm funtion to use the parse method instead of create

In [24]:
def llm_structured(user_prompt, output_type,instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [ ]:
## lets test with calendar event first and check if it works as expected

response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)


In [26]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [27]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [28]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )


In [29]:
answer = rag_structured('how do I do llm evals?')


In [32]:
print(answer.answer[:100])
print(answer.found_answer)

To perform LLM evaluations, you can follow these steps:

1. **Installation**: Install the Evidently 
True


In [33]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The provided context does not include information on how to install Kafka on Windows. Please refer t
False


In [36]:
# Making the Answer Optional

# Right now, when the answer isn't found, the LLM still generates text for the answer field.
#  Let's make it optional so it doesn't have to:
from typing import Optional
from pydantic import Field  

class RAGResponse(BaseModel):
    """
   The response from the RAG system. 
   if the answer is not found the answer field will be None and found_answer will be False
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")



In [37]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [ ]:
##lets addmore complexity to the question and see how it handles it

In [38]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [39]:
# This model adds:
# confidence - a score from 0.0 to 1.0
# confidence_explanation - why the LLM is this confident
# answer_type - categorizes the response (how-to, explanation, troubleshooting, comparison, reference)
# followup_questions - suggested next questions
# Notice that answer_type uses Literal - it can ONLY be one of these five specific values. The LLM cannot choose anything else.

In [40]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [41]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [42]:
answer

RAGResponse(answer='To evaluate LLM outputs, you can use the following approach by utilizing multiple LLMs as judges:\n\n### Steps to Evaluate LLMs\n1. **Prepare Your Environment**  \n   Install Evidently:\n   ```bash\n   pip install evidently litellm \n   ```  \n   (You can also install `evidently[llm]`.)\n\n2. **Set Up Evaluator LLMs**  \n   Pass the API keys for the LLMs that you will use:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR KEY"\n   os.environ["GEMINI_API_KEY"] = "YOUR KEY"\n   os.environ["ANTHROPIC_API_KEY"] = "YOUR KEY"\n   ```\n\n3. **Define the Dataset**  \n   Create a dataset of user intents and generated outputs. For example:\n   ```python\n   data = [\n       ["don’t want to attend, say no", "Hey, going to skip the meeting..."],\n       ...\n   ]\n   columns = ["user input", "generated email"]\n   eval_df = pd.DataFrame(data, columns=columns)\n   ```\n\n4. **Define the Evaluation Prompt**  \n   Use a `BinaryClassificationPromptTemplate` to se